# SEC 10-K Sparse Autoencoder (SAE) — Domain-Specific Feature Discovery

Train a production-quality SAE on GPT-2's internal activations from 10 years of SEC 10-K filings (S&P 500 universe) to surface financially meaningful latent features.

**Pipeline overview**
```
10-K filing text
  → clean & chunk (1024-token windows)
  → GPT-2 forward pass → MLP-L8 activations  [d_mlp = 3072]
  → HDF5 on-disk cache
  → SAE training  (dictionary size = 4 × 3072 = 12288)
  → feature interpretation & financial analysis
```

**Sections**
0. Config & Imports  
1. Data Loading (flexible: CSV / parquet / directory / JSONL)  
2. Text Preprocessing & Chunking  
3. Activation Extraction & HDF5 Caching  
4. Production SAE Architecture  
5. Training Run & Quality Diagnostics  
6. Feature Interpretation  
7. Section-Based Analysis (Risk Factors vs MD&A)  
8. Temporal Analysis (crisis vs normal years)  
9. Sector Analysis (GICS 11 sectors)  
10. Feature Steering & Concept Arithmetic  
11. Company / Filing-Level Profiles  
12. Downstream Task Validation  
13. Artifact Saving & Reproducibility Report  

## 0 · Config & Imports

In [ ]:
import os, re, json, math, time, random, logging, warnings, hashlib
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import h5py
from transformer_lens import HookedTransformer

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn not installed — UMAP visualisations will be skipped.")

try:
    import pyarrow.parquet as pq
    HAS_PARQUET = True
except ImportError:
    HAS_PARQUET = False

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── device ───────────────────────────────────────────────────────────────────
device = (
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

In [ ]:
@dataclass
class Config:
    # ── data ─────────────────────────────────────────────────────────────────
    # Set ONE of these to point at your 10-K corpus:
    data_csv:       Optional[str] = None          # path to CSV/parquet with columns: ticker, year, section, text
    data_dir:       Optional[str] = None          # root dir;  subdirs = tickers;  files = <year>_<section>.txt
    data_jsonl:     Optional[str] = None          # JSONL, one record per line: {"ticker", "year", "section", "text"}

    # ── model & hook ─────────────────────────────────────────────────────────
    model_name:     str  = 'gpt2'                 # any TransformerLens model name
    hook_layer:     int  = 8                      # which MLP output to intercept
    hook_name:      str  = 'blocks.{L}.hook_mlp_out'
    context_len:    int  = 1024
    chunk_stride:   int  = 512                    # overlap between chunks

    # ── activation cache ─────────────────────────────────────────────────────
    cache_path:     str  = 'activations_10k.h5'
    cache_batch:    int  = 8                      # chunks per forward pass

    # ── SAE ──────────────────────────────────────────────────────────────────
    expansion:      int  = 4                      # n_features = expansion × d_mlp
    l1_coeff:       float = 8e-4                  # initial L1 regularisation weight
    l1_warmup_steps: int = 1000                   # ramp L1 from 0 over this many steps
    ghost_threshold: float = 1e-3                 # min activation freq before ghost gradient kicks in
    resample_every: int  = 2500                   # steps between dead-neuron resampling
    resample_window: int = 500                    # track activation over last N steps for dead detection

    # ── training ─────────────────────────────────────────────────────────────
    sae_epochs:     int  = 3
    sae_batch:      int  = 1024
    sae_lr:         float = 3e-4
    sae_wd:         float = 0.0
    grad_clip:      float = 1.0

    # ── interpretation ───────────────────────────────────────────────────────
    top_k_tokens:   int  = 20                     # tokens shown per feature
    n_interp_features: int = 50                   # features to auto-label

    # ── output ───────────────────────────────────────────────────────────────
    artifact_dir:   str  = 'sec_sae_artifacts'

cfg = Config()

# ─── EDIT HERE: point at your data ───────────────────────────────────────────
# Example 1 – CSV
#   cfg.data_csv = '/path/to/10k_filings.csv'
# Example 2 – directory tree
#   cfg.data_dir = '/path/to/filings_root'
# Example 3 – JSONL
#   cfg.data_jsonl = '/path/to/10k_filings.jsonl'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(cfg.artifact_dir, exist_ok=True)
print("Config ready. Set cfg.data_csv / cfg.data_dir / cfg.data_jsonl to your corpus.")
print(json.dumps(asdict(cfg), indent=2))

## 1 · Data Loading

In [ ]:
from dataclasses import dataclass as _dc

@_dc
class FilingRecord:
    ticker:  str
    year:    int
    section: str   # e.g. 'risk_factors', 'mda', 'business'
    text:    str


class FilingLoader:
    """Loads 10-K filings from three flexible source formats."""

    @staticmethod
    def from_csv(path: str) -> List[FilingRecord]:
        """CSV / Parquet with columns: ticker, year, section, text."""
        p = Path(path)
        if p.suffix in ('.parquet', '.pq') and HAS_PARQUET:
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
        required = {'ticker', 'year', 'section', 'text'}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"CSV missing columns: {missing}")
        return [
            FilingRecord(str(r.ticker), int(r.year), str(r.section), str(r.text))
            for r in df.itertuples()
        ]

    @staticmethod
    def from_dir(root: str) -> List[FilingRecord]:
        """
        Directory layout:
          <root>/<TICKER>/<YEAR>_<section>.txt
          e.g.  filings/AAPL/2023_risk_factors.txt
        """
        records = []
        for ticker_dir in sorted(Path(root).iterdir()):
            if not ticker_dir.is_dir():
                continue
            ticker = ticker_dir.name
            for fpath in sorted(ticker_dir.glob('*.txt')):
                stem = fpath.stem  # e.g. '2022_mda'
                parts = stem.split('_', 1)
                if len(parts) != 2 or not parts[0].isdigit():
                    continue
                year, section = int(parts[0]), parts[1]
                records.append(FilingRecord(ticker, year, section, fpath.read_text()))
        return records

    @staticmethod
    def from_jsonl(path: str) -> List[FilingRecord]:
        """JSONL, one JSON object per line: {ticker, year, section, text}."""
        records = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                records.append(FilingRecord(
                    str(d['ticker']), int(d['year']), str(d['section']), str(d['text'])
                ))
        return records

    @classmethod
    def load(cls, cfg: Config) -> List[FilingRecord]:
        if cfg.data_csv:
            records = cls.from_csv(cfg.data_csv)
        elif cfg.data_dir:
            records = cls.from_dir(cfg.data_dir)
        elif cfg.data_jsonl:
            records = cls.from_jsonl(cfg.data_jsonl)
        else:
            raise ValueError("Set cfg.data_csv, cfg.data_dir, or cfg.data_jsonl")
        log.info(f"Loaded {len(records):,} filing records")
        return records


# ── demo: synthetic corpus so notebook runs end-to-end without real data ──────
def make_demo_corpus(n_companies=10, n_years=5) -> List[FilingRecord]:
    """Generates a tiny synthetic corpus for testing."""
    tickers = ['AAPL','MSFT','JPM','GS','XOM','CVX','JNJ','PFE','AMZN','GOOGL']
    sections = ['risk_factors', 'mda', 'business']
    templates = {
        'risk_factors': (
            "{ticker} faces significant risks including interest rate volatility, credit exposure, "
            "regulatory compliance requirements, cybersecurity threats, and macroeconomic uncertainty. "
            "In {year}, the company identified {n} material risk factors that could adversely affect "
            "operating results and financial condition. Competitive pressures in the industry remain intense."
        ),
        'mda': (
            "Management's discussion: {ticker} reported revenue of ${rev}B for fiscal year {year}, "
            "representing a {chg}% change year-over-year. Net income was ${ni}B. "
            "The company continued to invest in research and development, capital expenditures, "
            "and strategic acquisitions to drive long-term shareholder value. "
            "Operating margin expanded by {margin} basis points driven by efficiency improvements."
        ),
        'business': (
            "{ticker} is a leading provider of products and services across multiple segments. "
            "Founded in the early {decade}s, the company operates in {n_countries} countries worldwide "
            "with approximately {employees}k employees. The company's competitive advantages include "
            "brand recognition, intellectual property, distribution network, and customer loyalty."
        )
    }
    records = []
    rng = np.random.default_rng(0)
    for ticker in tickers[:n_companies]:
        for year in range(2014, 2014 + n_years):
            for section in sections:
                text = (templates[section] * 30).format(
                    ticker=ticker, year=year,
                    rev=round(rng.uniform(10, 400), 1),
                    ni=round(rng.uniform(1, 80), 1),
                    chg=round(rng.uniform(-10, 30), 1),
                    margin=int(rng.uniform(-50, 200)),
                    n=int(rng.integers(5, 25)),
                    decade=int(rng.integers(1960, 1995)),
                    n_countries=int(rng.integers(10, 150)),
                    employees=int(rng.integers(5, 500)),
                )
                records.append(FilingRecord(ticker, year, section, text))
    return records


# Load real data if configured, else fall back to demo corpus
try:
    filings = FilingLoader.load(cfg)
except ValueError:
    print("⚠  No data source configured — using synthetic demo corpus.")
    print("   Set cfg.data_csv / cfg.data_dir / cfg.data_jsonl to use real 10-K data.")
    filings = make_demo_corpus()

print(f"Total filings: {len(filings):,}")
tickers_all  = sorted({f.ticker  for f in filings})
years_all    = sorted({f.year    for f in filings})
sections_all = sorted({f.section for f in filings})
print(f"  Tickers : {len(tickers_all)}  |  Years: {years_all[0]}–{years_all[-1]}  |  Sections: {sections_all}")

## 2 · Text Preprocessing & Chunking

In [ ]:
from dataclasses import dataclass as _dc2

@_dc2
class ChunkRecord:
    ticker:     str
    year:       int
    section:    str
    chunk_idx:  int
    tokens:     List[int]   # token IDs, len <= cfg.context_len


def clean_10k_text(text: str) -> str:
    """Remove HTML tags, normalize whitespace, strip boilerplate headers."""
    # strip HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # remove EDGAR filing headers/footers
    text = re.sub(r'(?m)^-{5,}.*$', '', text)
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text)
    # remove page numbers  e.g. "F-12", "Page 45 of 120"
    text = re.sub(r'\bPage\s+\d+\s+of\s+\d+\b', '', text, flags=re.I)
    text = re.sub(r'\bF-\d+\b', '', text)
    return text.strip()


def chunk_filing(record: FilingRecord, model, context_len: int, stride: int) -> List[ChunkRecord]:
    """Tokenise and split into overlapping context windows."""
    text = clean_10k_text(record.text)
    token_ids = model.to_tokens(text, prepend_bos=True)[0].tolist()  # [seq]
    chunks = []
    start = 0
    idx   = 0
    while start < len(token_ids):
        end = start + context_len
        window = token_ids[start:end]
        if len(window) < 16:
            break  # skip tiny trailing windows
        chunks.append(ChunkRecord(
            ticker=record.ticker,
            year=record.year,
            section=record.section,
            chunk_idx=idx,
            tokens=window
        ))
        idx += 1
        start += stride
    return chunks


# ── load GPT-2 (for tokeniser + forward pass) ─────────────────────────────────
print("Loading GPT-2 small ...")
model = HookedTransformer.from_pretrained(cfg.model_name, device=device)
model.eval()

d_mlp      = model.cfg.d_mlp      # 3072 for GPT-2 small
n_features = cfg.expansion * d_mlp
HOOK_NAME  = cfg.hook_name.replace('{L}', str(cfg.hook_layer))
print(f"Hook point : {HOOK_NAME}   |  d_mlp = {d_mlp}  |  SAE features = {n_features:,}")

In [ ]:
print("Chunking all filings ...")
all_chunks: List[ChunkRecord] = []
for rec in tqdm(filings):
    all_chunks.extend(chunk_filing(rec, model, cfg.context_len, cfg.chunk_stride))

# stats
n_tokens_total = sum(len(c.tokens) for c in all_chunks)
print(f"Total chunks : {len(all_chunks):,}")
print(f"Total tokens : {n_tokens_total:,}  ({n_tokens_total/1e6:.1f}M)")
print(f"Avg chunk length : {n_tokens_total/len(all_chunks):.0f} tokens")

## 3 · Activation Extraction & HDF5 Caching

In [ ]:
class HDF5ActivationStore:
    """
    Stores MLP activations on disk as float16 HDF5.
    Layout:
      /activations   [N_total_tokens, d_mlp]  float16
      /metadata      JSON string per chunk     bytes
    """

    def __init__(self, path: str, d_mlp: int, mode: str = 'a'):
        self.path  = path
        self.d_mlp = d_mlp
        self.f     = h5py.File(path, mode)
        if 'activations' not in self.f:
            self.f.create_dataset(
                'activations',
                shape=(0, d_mlp),
                maxshape=(None, d_mlp),
                dtype='float16',
                chunks=(1024, d_mlp),
            )
            self.f.create_dataset(
                'metadata',
                shape=(0,),
                maxshape=(None,),
                dtype=h5py.string_dtype(),
            )

    def append(self, acts: np.ndarray, meta: dict):
        """acts: [seq_len, d_mlp] float32 → stored as float16."""
        n  = acts.shape[0]
        ds_a = self.f['activations']
        ds_m = self.f['metadata']
        old  = ds_a.shape[0]
        ds_a.resize((old + n, self.d_mlp))
        ds_m.resize((old + n,))
        ds_a[old:old+n] = acts.astype(np.float16)
        ds_m[old:old+n] = json.dumps(meta)

    def __len__(self):
        return self.f['activations'].shape[0]

    def close(self):
        self.f.close()


class ActivationExtractor:
    """Runs batched GPT-2 forward passes and stores MLP activations."""

    def __init__(self, model, hook_name: str, store: HDF5ActivationStore):
        self.model     = model
        self.hook_name = hook_name
        self.store     = store

    @torch.no_grad()
    def process_batch(self, chunks: List[ChunkRecord]):
        # pad to same length
        max_len = max(len(c.tokens) for c in chunks)
        pad_id  = self.model.tokenizer.eos_token_id or 0
        padded  = [
            c.tokens + [pad_id] * (max_len - len(c.tokens))
            for c in chunks
        ]
        tokens = torch.tensor(padded, device=device)  # [B, T]

        collected = {}
        def hook_fn(value, hook):
            collected['acts'] = value.cpu().float().numpy()  # [B, T, d_mlp]

        self.model.run_with_hooks(
            tokens,
            fwd_hooks=[(self.hook_name, hook_fn)],
        )
        acts = collected['acts']  # [B, T, d_mlp]

        for i, chunk in enumerate(chunks):
            seq_len = len(chunk.tokens)
            meta = dict(
                ticker=chunk.ticker,
                year=chunk.year,
                section=chunk.section,
                chunk_idx=chunk.chunk_idx,
                seq_len=seq_len,
            )
            self.store.append(acts[i, :seq_len], meta)

In [ ]:
cache_path = cfg.cache_path
store = HDF5ActivationStore(cache_path, d_mlp)

# Determine which chunks still need extraction (resumable)
already_stored = len(store)
# Each chunk contributes chunk.tokens tokens; we need to map stored token count → chunk index
# Simple approximation: skip first `already_stored` tokens worth of chunks
cumulative = 0
start_chunk_idx = 0
for i, chunk in enumerate(all_chunks):
    if cumulative >= already_stored:
        start_chunk_idx = i
        break
    cumulative += len(chunk.tokens)
else:
    start_chunk_idx = len(all_chunks)  # all done

remaining = all_chunks[start_chunk_idx:]
print(f"Already stored : {already_stored:,} token activations")
print(f"Chunks remaining : {len(remaining):,} of {len(all_chunks):,}")

extractor = ActivationExtractor(model, HOOK_NAME, store)
batch_size = cfg.cache_batch

for i in tqdm(range(0, len(remaining), batch_size), desc='Extracting activations'):
    batch = remaining[i : i + batch_size]
    extractor.process_batch(batch)
    if (i // batch_size) % 50 == 0:
        store.f.flush()

store.f.flush()
print(f"\nTotal activations stored : {len(store):,} token vectors  ({len(store) * d_mlp * 2 / 1e6:.1f} MB)")

## 4 · Production SAE Architecture

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Production SAE with:
    - Pre-encoder centering bias (b_pre) to handle mean-shift
    - Decoder columns unit-normalised (tied geometry)
    - Ghost gradient auxiliary loss for near-dead features
    """

    def __init__(self, d_in: int, n_features: int):
        super().__init__()
        self.d_in       = d_in
        self.n_features = n_features

        self.b_pre  = nn.Parameter(torch.zeros(d_in))          # centering
        self.W_enc  = nn.Parameter(torch.empty(d_in, n_features))
        self.b_enc  = nn.Parameter(torch.zeros(n_features))
        self.W_dec  = nn.Parameter(torch.empty(n_features, d_in))
        self.b_dec  = nn.Parameter(torch.zeros(d_in))

        nn.init.kaiming_uniform_(self.W_enc, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_dec, a=math.sqrt(5))
        self._normalise_decoder()

    @torch.no_grad()
    def _normalise_decoder(self):
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.div_(norms)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """x: [..., d_in]  →  hidden: [..., n_features]"""
        x_c = x - self.b_pre
        return F.relu(x_c @ self.W_enc + self.b_enc)

    def decode(self, h: torch.Tensor) -> torch.Tensor:
        return h @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        h    = self.encode(x)
        x_hat = self.decode(h)
        return x_hat, h

    def loss(self, x, x_hat, h, l1_coeff: float, ghost_mask: torch.Tensor = None):
        mse  = (x - x_hat).pow(2).mean()
        l1   = h.abs().sum(dim=-1).mean()
        loss = mse + l1_coeff * l1

        # Ghost gradient: nudge near-dead features toward high-loss residuals
        if ghost_mask is not None and ghost_mask.any():
            residual    = (x - x_hat).detach()
            ghost_acts  = F.relu((residual - self.b_pre) @ self.W_enc[:, ghost_mask] + self.b_enc[ghost_mask])
            ghost_recon = ghost_acts @ self.W_dec[ghost_mask]
            ghost_loss  = (residual - ghost_recon).pow(2).mean()
            loss = loss + 0.1 * ghost_loss

        return loss, mse, l1

In [ ]:
class DeadNeuronResampler:
    """Tracks per-feature activation frequency and resamples dead features."""

    def __init__(self, n_features: int, window: int = 500, threshold: float = 1e-3):
        self.n_features = n_features
        self.window     = window
        self.threshold  = threshold
        self.history    = torch.zeros(n_features)  # rolling activation count
        self.steps      = 0

    @torch.no_grad()
    def update(self, h: torch.Tensor):
        """h: [B, n_features]"""
        freq = (h > 0).float().mean(0).cpu()
        # exponential moving average
        alpha = 1.0 / self.window
        self.history = (1 - alpha) * self.history + alpha * freq
        self.steps  += 1

    def dead_mask(self) -> torch.Tensor:
        return self.history < self.threshold

    def ghost_mask(self) -> torch.Tensor:
        return (self.history > 0) & (self.history < self.threshold * 10)

    @torch.no_grad()
    def resample(self, sae: SparseAutoencoder, high_loss_acts: torch.Tensor):
        """Replace dead encoder directions with rescaled high-loss input vectors."""
        dead = self.dead_mask()
        n_dead = dead.sum().item()
        if n_dead == 0:
            return 0
        # sample n_dead rows from high_loss_acts
        idx = torch.randint(0, high_loss_acts.shape[0], (n_dead,))
        new_dirs = high_loss_acts[idx].to(sae.W_enc.device)  # [n_dead, d_in]
        # normalize
        new_dirs = F.normalize(new_dirs, dim=-1)
        sae.W_enc[:, dead] = new_dirs.T
        sae.W_dec[dead]    = new_dirs
        sae.b_enc.data[dead] = 0.0
        # reset history for resampled features
        self.history[dead] = self.threshold * 2
        return n_dead


class AdaptiveL1Scheduler:
    """Warms L1 coefficient up from 0 over l1_warmup_steps."""

    def __init__(self, base: float, warmup_steps: int):
        self.base   = base
        self.warmup = warmup_steps

    def get(self, step: int) -> float:
        if step >= self.warmup:
            return self.base
        return self.base * step / self.warmup


class HDF5StreamingDataset(Dataset):
    """Lazily reads activation vectors from the HDF5 cache."""

    def __init__(self, path: str):
        self.path = path
        with h5py.File(path, 'r') as f:
            self.n = f['activations'].shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        with h5py.File(self.path, 'r') as f:
            x = f['activations'][idx].astype(np.float32)
        return torch.from_numpy(x)


print(f"SAE: d_in={d_mlp}  n_features={n_features:,}")
sae = SparseAutoencoder(d_mlp, n_features).to(device)
print(f"SAE parameters: {sum(p.numel() for p in sae.parameters()):,}")

## 5 · Training Run & Quality Diagnostics

In [ ]:
store.close()  # close extractor handle so Dataset can open it

dataset    = HDF5StreamingDataset(cfg.cache_path)
dataloader = DataLoader(
    dataset,
    batch_size=cfg.sae_batch,
    shuffle=True,
    num_workers=0,
    pin_memory=(device == 'cuda'),
)

optimizer  = torch.optim.Adam(sae.parameters(), lr=cfg.sae_lr, weight_decay=cfg.sae_wd)
resampler  = DeadNeuronResampler(n_features, cfg.resample_window, cfg.ghost_threshold)
l1_sched   = AdaptiveL1Scheduler(cfg.l1_coeff, cfg.l1_warmup_steps)

history    = {'step': [], 'loss': [], 'mse': [], 'l1': [], 'l0': [], 'dead_frac': []}
step       = 0
high_loss_buffer = []   # stores high-loss activations for resampling

sae.train()
for epoch in range(cfg.sae_epochs):
    ep_loss = 0.0
    for batch in tqdm(dataloader, desc=f'Epoch {epoch+1}/{cfg.sae_epochs}'):
        x = batch.to(device)  # [B, d_mlp]

        l1_coeff    = l1_sched.get(step)
        ghost_mask  = resampler.ghost_mask().to(device)

        x_hat, h = sae(x)
        loss, mse, l1 = sae.loss(x, x_hat, h, l1_coeff, ghost_mask)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(sae.parameters(), cfg.grad_clip)
        optimizer.step()
        sae._normalise_decoder()

        resampler.update(h.detach())

        # collect high-loss samples for resampling
        with torch.no_grad():
            per_sample_loss = (x - x_hat).pow(2).mean(-1)  # [B]
            high_idx = per_sample_loss.topk(min(8, x.shape[0])).indices
            high_loss_buffer.append(x[high_idx].cpu())
            if len(high_loss_buffer) > 50:
                high_loss_buffer.pop(0)

        # dead neuron resampling
        if step > 0 and step % cfg.resample_every == 0 and high_loss_buffer:
            pool = torch.cat(high_loss_buffer, 0)
            n_resampled = resampler.resample(sae, pool)
            if n_resampled:
                log.info(f"Step {step}: resampled {n_resampled} dead features")

        # logging
        if step % 100 == 0:
            with torch.no_grad():
                l0    = (h > 0).float().sum(-1).mean().item()
                dead  = resampler.dead_mask().float().mean().item()
            history['step'].append(step)
            history['loss'].append(loss.item())
            history['mse'].append(mse.item())
            history['l1'].append(l1.item())
            history['l0'].append(l0)
            history['dead_frac'].append(dead)

        ep_loss += loss.item()
        step    += 1

    print(f"Epoch {epoch+1} avg loss: {ep_loss/len(dataloader):.4f}")

sae.eval()
print("Training complete.")

In [ ]:
# ── training diagnostics ──────────────────────────────────────────────────────
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, col, title in zip(
    axes.flatten(),
    ['loss', 'mse', 'l1', 'l0', 'dead_frac'],
    ['Total Loss', 'Reconstruction MSE', 'L1 Sparsity', 'L0 (avg active features)', 'Dead Feature Fraction']
):
    ax.plot(hist_df['step'], hist_df[col])
    ax.set_title(title)
    ax.set_xlabel('step')
axes[-1, -1].axis('off')
plt.suptitle('SAE Training Diagnostics', fontsize=14)
plt.tight_layout()
plt.savefig(f"{cfg.artifact_dir}/training_diagnostics.png", dpi=150)
plt.show()

# ── explained variance ────────────────────────────────────────────────────────
print("\nFinal metrics:")
print(f"  L0   (avg features active) : {hist_df['l0'].iloc[-1]:.1f} / {n_features}")
print(f"  Dead feature fraction       : {hist_df['dead_frac'].iloc[-1]*100:.1f}%")
print(f"  Final MSE                   : {hist_df['mse'].iloc[-1]:.4f}")

## 6 · Feature Interpretation

In [ ]:
# Re-open HDF5 store (read-only) and build a small interpretation sample
MAX_INTERP_SAMPLES = 20_000  # cap for speed

with h5py.File(cfg.cache_path, 'r') as f:
    n_stored = f['activations'].shape[0]
    idx_sample = np.random.choice(n_stored, min(MAX_INTERP_SAMPLES, n_stored), replace=False)
    idx_sample.sort()
    acts_np  = f['activations'][idx_sample].astype(np.float32)  # [N, d_mlp]
    meta_raw = [json.loads(f['metadata'][i]) for i in idx_sample]

acts_tensor = torch.from_numpy(acts_np).to(device)

with torch.no_grad():
    _, hidden = sae(acts_tensor)  # [N, n_features]

hidden_np = hidden.cpu().numpy()
print(f"Hidden matrix : {hidden_np.shape}  (samples × features)")
print(f"Mean L0       : {(hidden_np > 0).sum(-1).mean():.1f}")

In [ ]:
def get_top_activating_tokens(feature_idx: int, top_k: int = 20) -> List[Dict]:
    """
    Returns the top-k samples (with their metadata) where feature `feature_idx`
    has the highest activation.
    """
    acts_col = hidden_np[:, feature_idx]     # [N]
    top_idx  = np.argsort(acts_col)[-top_k:][::-1]
    results  = []
    for i in top_idx:
        results.append({
            'activation' : float(acts_col[i]),
            'ticker'     : meta_raw[i]['ticker'],
            'year'       : meta_raw[i]['year'],
            'section'    : meta_raw[i]['section'],
        })
    return results


FINANCIAL_CONCEPT_KEYWORDS = {
    'liquidity_risk'    : ['liquidity','cash','funding','credit facility','revolving'],
    'market_risk'       : ['interest rate','foreign exchange','market risk','hedge','derivative'],
    'credit_risk'       : ['credit risk','default','counterparty','allowance','impairment'],
    'regulatory_risk'   : ['regulation','compliance','SEC','FINRA','Dodd-Frank','capital requirement'],
    'revenue_growth'    : ['revenue','sales growth','net sales','organic growth'],
    'profitability'     : ['net income','operating income','EBITDA','margin','earnings'],
    'capex'             : ['capital expenditure','capex','property plant','investment'],
    'debt_leverage'     : ['debt','leverage','long-term debt','notes payable','credit rating'],
    'cybersecurity'     : ['cybersecurity','data breach','information security','ransomware'],
    'esg'               : ['sustainability','ESG','climate','carbon','environmental'],
    'ma_activity'       : ['acquisition','merger','divestiture','goodwill','intangible'],
    'litigation'        : ['litigation','lawsuit','legal proceedings','settlement','judgment'],
}


def auto_label_feature(feature_idx: int, top_k: int = 20) -> str:
    """Heuristic labeling: look for keyword overlap in top-activating section names."""
    top_samples = get_top_activating_tokens(feature_idx, top_k)
    sections    = [s['section'].lower() for s in top_samples]
    section_str = ' '.join(sections)

    scores = {}
    for concept, keywords in FINANCIAL_CONCEPT_KEYWORDS.items():
        scores[concept] = sum(kw in section_str for kw in keywords)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'unknown'


# ── compute feature summary stats ─────────────────────────────────────────────
mean_acts   = hidden_np.mean(0)    # [n_features]
max_acts    = hidden_np.max(0)     # [n_features]
freq_acts   = (hidden_np > 0).mean(0)  # [n_features]

feature_df = pd.DataFrame({
    'feature_idx' : np.arange(n_features),
    'mean_act'    : mean_acts,
    'max_act'     : max_acts,
    'freq'        : freq_acts,
})
feature_df = feature_df.sort_values('mean_act', ascending=False).reset_index(drop=True)

print(feature_df.head(10).to_string())

In [ ]:
# Auto-label top-N features
n_label = cfg.n_interp_features
top_features = feature_df['feature_idx'].values[:n_label]

labels = {}
for fi in tqdm(top_features, desc='Labeling features'):
    labels[fi] = auto_label_feature(fi)

feature_df['label'] = feature_df['feature_idx'].map(labels).fillna('unlabeled')

# Show top features per financial concept
print("\nTop labeled features by concept:")
print(feature_df[feature_df['label'] != 'unlabeled'].groupby('label')['mean_act'].mean().sort_values(ascending=False))

# ── feature activation distribution ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(freq_acts, bins=50, color='steelblue', edgecolor='white')
ax1.set_xlabel('Activation frequency')
ax1.set_ylabel('Number of features')
ax1.set_title('Feature activation frequency distribution')

ax2.hist(np.log10(max_acts + 1e-8), bins=50, color='coral', edgecolor='white')
ax2.set_xlabel('log10(max activation)')
ax2.set_title('Feature max-activation distribution')

plt.tight_layout()
plt.savefig(f"{cfg.artifact_dir}/feature_distributions.png", dpi=150)
plt.show()

# ── sample feature deep-dive ──────────────────────────────────────────────────
DEMO_FEATURE = int(top_features[0])
print(f"\n── Feature #{DEMO_FEATURE} (label: {labels.get(DEMO_FEATURE, '?')}) ──")
top_samples = get_top_activating_tokens(DEMO_FEATURE, cfg.top_k_tokens)
for rank, s in enumerate(top_samples):
    print(f"  {rank+1:2d}. act={s['activation']:.3f}  {s['ticker']} {s['year']}  [{s['section']}]")

## 7 · Section-Based Analysis (Risk Factors vs MD&A)

In [ ]:
# Compute mean feature activation per filing section
sections_present = sorted({m['section'] for m in meta_raw})
section_means = {}

for sec in sections_present:
    idx_sec = [i for i, m in enumerate(meta_raw) if m['section'] == sec]
    if idx_sec:
        section_means[sec] = hidden_np[idx_sec].mean(0)  # [n_features]

# Show heatmap for top-50 features × sections
TOP_FEAT = 50
feat_order = feature_df['feature_idx'].values[:TOP_FEAT]
mat = np.array([section_means[s][feat_order] for s in sections_present])  # [n_sec, TOP_FEAT]

# Row-normalise for comparison
mat_norm = mat / (mat.max(1, keepdims=True) + 1e-8)

fig = px.imshow(
    mat_norm,
    x=[f'F{i}' for i in feat_order],
    y=sections_present,
    color_continuous_scale='RdBu_r',
    title='Mean feature activation by 10-K section (top-50 features, row-normalised)',
    aspect='auto',
    height=300 + 60 * len(sections_present),
)
fig.show()
fig.write_html(f"{cfg.artifact_dir}/section_heatmap.html")

In [ ]:
def section_specificity_score(feature_idx: int) -> Dict[str, float]:
    """
    Returns a per-section specificity score:
    how much higher does this feature activate in this section vs the mean?
    (positive = section-specific, negative = suppressed)
    """
    overall_mean = hidden_np[:, feature_idx].mean()
    scores = {}
    for sec, mu in section_means.items():
        scores[sec] = float(mu[feature_idx] - overall_mean)
    return scores


# show for demo feature
spec = section_specificity_score(DEMO_FEATURE)
print(f"Section specificity for feature #{DEMO_FEATURE}:")
for sec, val in sorted(spec.items(), key=lambda x: -x[1]):
    bar = '█' * int(max(0, val * 50 + 0.5))
    print(f"  {sec:20s}  {val:+.4f}  {bar}")

## 8 · Temporal Analysis

In [ ]:
years_present = sorted({m['year'] for m in meta_raw})

# Mean feature activation per year
year_means = {}
for yr in years_present:
    idx_yr = [i for i, m in enumerate(meta_raw) if m['year'] == yr]
    if idx_yr:
        year_means[yr] = hidden_np[idx_yr].mean(0)  # [n_features]

# Feature temporal change score: std of yearly means (volatile = high score)
yearly_mat = np.array([year_means[y] for y in years_present])  # [n_years, n_features]
temporal_change = yearly_mat.std(0)  # [n_features]
feature_df['temporal_change'] = temporal_change

# Top temporally changing features
top_temporal = feature_df.nlargest(20, 'temporal_change')[['feature_idx','label','mean_act','temporal_change']]
print("Top 20 temporally volatile features:")
print(top_temporal.to_string())

In [ ]:
# ── Volcano plot: mean activation vs temporal change ──────────────────────────
fig = px.scatter(
    feature_df.head(500),  # top-500 by mean
    x='mean_act',
    y='temporal_change',
    color='label',
    hover_data=['feature_idx'],
    title='Volcano plot: mean activation vs temporal volatility (top-500 features)',
    labels={'mean_act': 'Mean activation', 'temporal_change': 'Temporal change (std across years)'},
)
fig.show()
fig.write_html(f"{cfg.artifact_dir}/volcano_plot.html")

# ── year-trajectory for top temporally volatile feature ───────────────────────
volatile_feat = int(feature_df.nlargest(1, 'temporal_change')['feature_idx'].iloc[0])
y_vals = [year_means[y][volatile_feat] for y in years_present]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=years_present, y=y_vals, mode='lines+markers', name=f'Feature #{volatile_feat}'))
fig2.update_layout(
    title=f'Feature #{volatile_feat} ({feature_df.set_index("feature_idx")["label"].get(volatile_feat, "?")}) — year trajectory',
    xaxis_title='Year', yaxis_title='Mean activation'
)
fig2.show()
fig2.write_html(f"{cfg.artifact_dir}/temporal_trajectory.html")

In [ ]:
# ── crisis vs normal analysis ─────────────────────────────────────────────────
# Designate crisis years (2008-2009, 2020) vs normal years
CRISIS_YEARS  = {2008, 2009, 2020}
crisis_idx    = [i for i, m in enumerate(meta_raw) if m['year'] in CRISIS_YEARS]
normal_idx    = [i for i, m in enumerate(meta_raw) if m['year'] not in CRISIS_YEARS]

if crisis_idx and normal_idx:
    crisis_mean = hidden_np[crisis_idx].mean(0)   # [n_features]
    normal_mean = hidden_np[normal_idx].mean(0)
    diff        = crisis_mean - normal_mean

    # Top features that activate MORE during crisis
    top_crisis_feat = np.argsort(diff)[-15:][::-1]
    print("Features most elevated during crisis years:")
    for fi in top_crisis_feat:
        lbl = feature_df.set_index('feature_idx')['label'].get(fi, '?')
        print(f"  Feature {fi:5d}  delta={diff[fi]:+.4f}  label={lbl}")
else:
    print("No crisis-year samples in current corpus (demo corpus has 2014–2018).")

## 9 · Sector Analysis (GICS 11 Sectors)

In [ ]:
# GICS sector mapping — extend / replace with your actual ticker→sector lookup
GICS_MAP = {
    'AAPL': 'Information Technology', 'MSFT': 'Information Technology',
    'GOOGL':'Information Technology', 'AMZN': 'Consumer Discretionary',
    'JPM':  'Financials',             'GS':   'Financials',
    'XOM':  'Energy',                 'CVX':  'Energy',
    'JNJ':  'Health Care',            'PFE':  'Health Care',
    # add more tickers here …
}

def get_sector(ticker: str) -> str:
    return GICS_MAP.get(ticker.upper(), 'Unknown')

sectors_in_data = sorted({get_sector(m['ticker']) for m in meta_raw})

sector_means = {}
for sec in sectors_in_data:
    idx_sec = [i for i, m in enumerate(meta_raw) if get_sector(m['ticker']) == sec]
    if idx_sec:
        sector_means[sec] = hidden_np[idx_sec].mean(0)

print(f"Sectors in corpus: {sectors_in_data}")

In [ ]:
# ── sector overlap heatmap ────────────────────────────────────────────────────
# Cosine similarity between sector feature activation profiles
sec_names = [s for s in sectors_in_data if s in sector_means]
sec_mat   = np.array([sector_means[s][:n_features] for s in sec_names])  # [n_sec, n_features]

norms     = np.linalg.norm(sec_mat, axis=1, keepdims=True) + 1e-8
sec_normed = sec_mat / norms
sim_mat   = sec_normed @ sec_normed.T  # [n_sec, n_sec]

fig = px.imshow(
    sim_mat,
    x=sec_names, y=sec_names,
    color_continuous_scale='Viridis',
    title='Sector feature-profile similarity (cosine)',
    zmin=0, zmax=1,
)
fig.show()
fig.write_html(f"{cfg.artifact_dir}/sector_similarity.html")

# ── features most sector-specific ─────────────────────────────────────────────
if len(sec_names) >= 2:
    sec_stack = np.array([sector_means[s] for s in sec_names])  # [n_sec, n_features]
    sec_var   = sec_stack.var(0)  # variance across sectors — high = sector-discriminating
    top_sector_feat = np.argsort(sec_var)[-10:][::-1]
    print("Top sector-discriminating features:")
    for fi in top_sector_feat:
        print(f"  Feature {fi:5d}  sector_var={sec_var[fi]:.4f}")

## 10 · Feature Steering & Concept Arithmetic

In [ ]:
def steer_activation(
    text: str,
    feature_idx: int,
    steer_value: float,
    hook_layer: int = None,
) -> Dict:
    """
    Run GPT-2 on `text`, then clamp feature `feature_idx` in the SAE latent space
    to `steer_value`, decode back, patch into MLP output, and get new predictions.

    Returns:
        original_top5   : list of (token, prob) before steering
        steered_top5    : list of (token, prob) after steering
    """
    hook_layer = hook_layer or cfg.hook_layer
    hook_name  = cfg.hook_name.replace('{L}', str(hook_layer))

    tokens = model.to_tokens(text, prepend_bos=True)

    with torch.no_grad():
        # baseline
        logits_orig = model(tokens)
        probs_orig  = logits_orig[0, -1].softmax(-1)

    original_top5 = [
        (model.to_string(i), float(probs_orig[i]))
        for i in probs_orig.topk(5).indices
    ]

    steered_logits = [None]

    def steering_hook(value, hook):
        # value: [1, seq, d_mlp]
        mlp_out   = value[0]  # [seq, d_mlp]
        _, hidden = sae(mlp_out)
        # clamp feature
        hidden_steered             = hidden.clone()
        hidden_steered[:, feature_idx] = steer_value
        recon_steered              = sae.decode(hidden_steered)
        return recon_steered.unsqueeze(0)

    with torch.no_grad():
        logits_steered = model.run_with_hooks(
            tokens,
            fwd_hooks=[(hook_name, steering_hook)],
        )
    probs_steered = logits_steered[0, -1].softmax(-1)

    steered_top5 = [
        (model.to_string(i), float(probs_steered[i]))
        for i in probs_steered.topk(5).indices
    ]

    return {'original_top5': original_top5, 'steered_top5': steered_top5}


# ── demo steering ─────────────────────────────────────────────────────────────
steer_text   = "The company faces material risks related to"
steer_result = steer_activation(steer_text, DEMO_FEATURE, steer_value=5.0)

print(f"Steering feature #{DEMO_FEATURE} to 5.0 on:\n  '{steer_text}'\n")
print("Original top-5 next tokens:")
for tok, prob in steer_result['original_top5']:
    print(f"  {tok!r:20s} {prob:.3%}")
print("\nSteered top-5 next tokens:")
for tok, prob in steer_result['steered_top5']:
    print(f"  {tok!r:20s} {prob:.3%}")

In [ ]:
# ── concept arithmetic ────────────────────────────────────────────────────────
# Find features for two financial concepts and compute their decoder directions

def concept_direction(concept_label: str) -> torch.Tensor:
    """Returns the mean decoder direction for all features with a given label."""
    feat_ids = feature_df[feature_df['label'] == concept_label]['feature_idx'].values
    if len(feat_ids) == 0:
        raise ValueError(f"No features labeled '{concept_label}'")
    with torch.no_grad():
        dec_vecs = sae.W_dec[feat_ids]  # [k, d_mlp]
    return dec_vecs.mean(0)  # [d_mlp]


unique_labels = [l for l in feature_df['label'].unique() if l not in ('unlabeled','unknown')]
print(f"Labeled concepts available: {unique_labels}")

if len(unique_labels) >= 2:
    dir_a = concept_direction(unique_labels[0])
    dir_b = concept_direction(unique_labels[1])
    cos_sim = F.cosine_similarity(dir_a.unsqueeze(0), dir_b.unsqueeze(0)).item()
    print(f"\nCosine similarity between '{unique_labels[0]}' and '{unique_labels[1]}' directions: {cos_sim:.4f}")
    print("(High similarity → concepts encoded in similar MLP directions)")

## 11 · Company / Filing-Level Profiles

In [ ]:
# Build per-filing feature profile (ticker × year × section)
filing_profiles = {}

from itertools import groupby
from operator import itemgetter

def filing_key(m): return (m['ticker'], m['year'], m['section'])
indexed = sorted(enumerate(meta_raw), key=lambda x: filing_key(x[1]))

for key, group in groupby(indexed, key=lambda x: filing_key(x[1])):
    idxs = [g[0] for g in group]
    filing_profiles[key] = hidden_np[idxs].mean(0)  # [n_features]

print(f"Filing profiles built: {len(filing_profiles)}")

# Year-over-year drift for a company
def company_yoy_drift(ticker: str, section: str = 'risk_factors') -> pd.DataFrame:
    years_t = sorted([
        yr for (t, yr, s) in filing_profiles
        if t == ticker and s == section
    ])
    if len(years_t) < 2:
        return pd.DataFrame()
    rows = []
    for i in range(1, len(years_t)):
        prev = filing_profiles[(ticker, years_t[i-1], section)]
        curr = filing_profiles[(ticker, years_t[i],   section)]
        diff = curr - prev
        top_up   = np.argsort(diff)[-5:][::-1]
        top_down = np.argsort(diff)[:5]
        rows.append({
            'ticker': ticker, 'year': years_t[i], 'section': section,
            'l2_drift': float(np.linalg.norm(diff)),
            'top_rising_features':  list(top_up),
            'top_falling_features': list(top_down),
        })
    return pd.DataFrame(rows)


demo_ticker = tickers_all[0]
demo_section = sections_all[0]
drift_df = company_yoy_drift(demo_ticker, demo_section)
if not drift_df.empty:
    print(f"\nYear-over-year feature drift for {demo_ticker} [{demo_section}]:")
    print(drift_df[['year','l2_drift','top_rising_features']].to_string())

    fig = px.bar(
        drift_df, x='year', y='l2_drift',
        title=f'{demo_ticker} feature drift (L2 norm of ΔFeatures) — {demo_section}',
        labels={'l2_drift': 'L2 drift', 'year': 'Year'}
    )
    fig.show()
else:
    print("Not enough years for drift analysis in this demo corpus.")

## 12 · Downstream Task Validation

In [ ]:
"""
Validate that SAE features encode financially meaningful signals by testing
whether they can predict an observable outcome.

Replace the synthetic targets below with real labels:
  - analyst_revision: +1 / -1 for EPS estimate revisions
  - volatility_regime: binary (high/low 12-month realized vol)
  - credit_rating_change: upgrade / stable / downgrade
"""

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

# Build X = per-filing mean SAE features, y = synthetic target
# (replace with real targets once your data pipeline is ready)
filing_keys   = list(filing_profiles.keys())
X_filing      = np.array([filing_profiles[k] for k in filing_keys])  # [n_filings, n_features]

# Synthetic target: companies in Financial / Energy sectors tend to have higher
# risk-factor activation (just for demonstration)
rng2 = np.random.default_rng(99)
y_synthetic = np.array([
    1 if get_sector(k[0]) in ('Financials', 'Energy') else 0
    for k in filing_keys
])

if y_synthetic.sum() > 5 and (1 - y_synthetic).sum() > 5:
    # Use top-200 features only (by mean activation) to keep regression stable
    top200 = feature_df['feature_idx'].values[:200]
    X_sub  = X_filing[:, top200]

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_sub)

    clf    = LogisticRegression(max_iter=500, C=0.1)
    scores = cross_val_score(clf, X_scaled, y_synthetic, cv=5, scoring='roc_auc')

    print(f"Downstream classification AUC (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}")
    print("(AUC > 0.6 suggests features carry economically meaningful signal)")
    print("\nNote: Replace y_synthetic with real financial labels for valid validation.")
else:
    print("Skipping downstream validation: insufficient class balance in demo corpus.")

## 13 · Artifact Saving & Reproducibility Report

In [ ]:
# ── save SAE weights ──────────────────────────────────────────────────────────
sae_path = f"{cfg.artifact_dir}/sae_weights.pt"
torch.save({
    'state_dict'   : sae.state_dict(),
    'd_in'         : d_mlp,
    'n_features'   : n_features,
    'hook_name'    : HOOK_NAME,
    'model_name'   : cfg.model_name,
}, sae_path)
print(f"SAE saved → {sae_path}")

# ── save feature dataframe ────────────────────────────────────────────────────
feat_path = f"{cfg.artifact_dir}/feature_profiles.parquet"
try:
    feature_df.to_parquet(feat_path, index=False)
    print(f"Feature profiles → {feat_path}")
except Exception:
    csv_path = feat_path.replace('.parquet', '.csv')
    feature_df.to_csv(csv_path, index=False)
    print(f"Feature profiles → {csv_path}")

# ── save filing profiles ──────────────────────────────────────────────────────
filing_rows = []
for (ticker, year, section), vec in filing_profiles.items():
    row = {'ticker': ticker, 'year': year, 'section': section}
    # store only top-200 features for compactness
    for rank, fi in enumerate(feature_df['feature_idx'].values[:200]):
        row[f'feat_{fi}'] = float(vec[fi])
    filing_rows.append(row)

filing_profile_df = pd.DataFrame(filing_rows)
fp_path = f"{cfg.artifact_dir}/filing_profiles.csv"
filing_profile_df.to_csv(fp_path, index=False)
print(f"Filing profiles → {fp_path}")

# ── reproducibility report ────────────────────────────────────────────────────
report = {
    'timestamp'       : time.strftime('%Y-%m-%dT%H:%M:%S'),
    'model'           : cfg.model_name,
    'hook_layer'      : cfg.hook_layer,
    'hook_name'       : HOOK_NAME,
    'd_mlp'           : d_mlp,
    'n_features'      : n_features,
    'expansion'       : cfg.expansion,
    'l1_coeff'        : cfg.l1_coeff,
    'sae_epochs'      : cfg.sae_epochs,
    'sae_batch'       : cfg.sae_batch,
    'sae_lr'          : cfg.sae_lr,
    'total_tokens'    : n_tokens_total,
    'total_chunks'    : len(all_chunks),
    'n_filings'       : len(filings),
    'n_filing_profiles': len(filing_profiles),
    'tickers'         : tickers_all,
    'years'           : years_all,
    'sections'        : sections_all,
    'final_mse'       : float(hist_df['mse'].iloc[-1]) if not hist_df.empty else None,
    'final_l0'        : float(hist_df['l0'].iloc[-1])  if not hist_df.empty else None,
    'dead_frac'       : float(hist_df['dead_frac'].iloc[-1]) if not hist_df.empty else None,
    'seed'            : SEED,
    'device'          : device,
}

report_path = f"{cfg.artifact_dir}/reproducibility_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"Reproducibility report → {report_path}")

print("\n" + "="*60)
print("  All artifacts saved.")
print("="*60)
print(json.dumps(report, indent=2))

---
## Loading a Saved SAE

```python
checkpoint = torch.load('sec_sae_artifacts/sae_weights.pt', map_location='cpu')
sae_loaded = SparseAutoencoder(checkpoint['d_in'], checkpoint['n_features'])
sae_loaded.load_state_dict(checkpoint['state_dict'])
sae_loaded.eval()
```

## Next Steps

1. **Point `cfg.data_csv` / `cfg.data_dir` / `cfg.data_jsonl`** at your real 10-K corpus and re-run from Section 2.
2. **Update `GICS_MAP`** (Section 9) with your full ticker→sector mapping.
3. **Replace `y_synthetic`** (Section 12) with real financial labels (analyst revisions, volatility, credit ratings).
4. **Scale the SAE**: increase `cfg.expansion` to 8 or 16 for richer features once you have sufficient GPU memory.
5. **Domain adaptation**: fine-tune GPT-2 on 10-K text before extracting activations for even more specialised features.
6. **Multi-layer SAE**: run the pipeline on multiple `hook_layer` values (e.g., 6, 8, 10) to capture different abstraction levels.
